In [1]:
import pandas as pd
import numpy as np
import os, ast
os.chdir('..')
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
f2 = 'data/OTS/'
outdir = 'data/output/figures/'

In [ ]:
all_seqs = pd.read_csv(f2 + 'OTS_all_studies_combined.csv.gz')

run_size = all_seqs['Run'].value_counts()
large_runs =  run_size > 100
all_runs = run_size.index.tolist()
studies = np.array(all_runs)[large_runs.tolist()]
print(len(studies))

print(studies)

In [ ]:
MI = pd.DataFrame()

for i, s in enumerate(studies):
    if i % 50 == 0: print('study ' + str(i) + ' of ' + str(len(studies)))
    try:
        df = pd.read_csv('data/output/mutual_info/OTS/mutual_info_' + s + '.csv')
        MI = pd.concat([MI, df])
    except: print('skipping study: ' + s); continue

In [5]:
MI = MI.set_index(['type', 'Unnamed: 0'])
MI.columns = [int(x) for x in MI.columns]
MI = MI.sort_index(axis=1)
# print(MI)

In [6]:
MI1 = MI.dropna(axis=0, how='all')
MI1 = MI1.replace(pd.NA, '[]')

In [ ]:
for c in MI1.columns:
    MI1[c] = [ast.literal_eval(x) for x in MI1[c]]
    MI1[c] = [x if type(x) == list else [x]*10 for x in MI1[c]]
print(MI1)

In [ ]:
MI1 = MI1.reset_index(names = ['type','epitope'])
MI1['shuffle'] = ['shuffle' if 'shuffle' in x else 'real' for x in MI1['epitope']]
MI1['epitope'] = [x.split('_')[0] for x in MI1['epitope']]
print(MI1)

In [9]:
MI2 = MI1.set_index(['epitope', 'shuffle', 'type']).apply(pd.Series).stack().explode()
MI2 = MI2.reset_index().rename(columns = {'level_3':'subsample', 0:'value'}).dropna(subset='value')

In [10]:
MI2['1/N'] = 1/MI2['subsample']

In [11]:
def fit_linear(mydf, title, ax):    
    single_vals = []
    def _fit(mydf1):
        df = mydf1.copy()
        df = df.dropna(subset=['value'])
        x = np.array(df['1/N'])
        myfit = np.polyfit(x, df['value'].astype('float64'), deg=1)
        p = np.poly1d(myfit)
        x_unique = np.linspace(0, max(x))
        y_unique = p(x_unique)
        # print(p, x_unique, y_unique)
        # print(x, df['value'].astype('float64'))
        assert p(0) == myfit[1]
        return(x_unique, y_unique, p(0))
    
    for j in ['real', 'shuffle']:
        if j == 'real':
            linestyle = '-'
            marker = 'o'
            s = 30
            col='r'
        else:
            linestyle = '-'
            marker = 'X'
            s = 30
            col='k'
        mydf0 = mydf.loc[mydf.shuffle == j]
        x_unique, y_unique, single_val = _fit(mydf0)    
        xy = pd.DataFrame([x_unique, y_unique], index = ['x', 'y']).T
        sns.scatterplot(data = mydf0, x = '1/N', y='value', ax=ax, alpha = 0.5, marker = marker, c=col, s = s, legend=None)
        sns.lineplot(data=xy, x='x', y='y', color = col, linestyle = linestyle, alpha = 0.5,  label = j, ax=ax)
        single_vals.append(single_val)
        # ax.scatter(0, single_val, marker = marker, color = col, s = 300, alpha = 0.5)
    ax.set_title(title)
    ax.set_xlim(0,0.041)
    # ax.set_yscale('log')
    # ax.set_xscale('log')

    return(single_vals)
    

In [ ]:
single_val = {}
s = MI2.shape
print(s)
print(MI2['type'].unique().tolist())
MI2_1 = MI2.dropna(subset='type')
print(MI2.shape)
assert s == MI2_1.shape

for ep in sorted(MI2['epitope'].unique().tolist()):
    print(ep)
    f, ax = plt.subplots(ncols=5, nrows=3, figsize=(12,8), sharex=True, sharey=True, constrained_layout=True)
    axs = ax.ravel()
    for i, t in enumerate(sorted(MI2['type'].unique().tolist())):
        # print(i, t)
        ss = MI2.loc[(MI2.epitope == ep) & (MI2.type == t)]
        # print(ss.sort_values(by='value', ascending=False))
        single_val[ep + '_' + t] = fit_linear(ss, t, axs[i])
        # print(single_val)
        axs[i].set_ylabel('')
        if i in [0,5,10]:
            axs[i].set_ylabel('MI (nats)')
        if i >= 10:
            axs[i].set_xlabel('1/N')
        
        h, l = axs[i].get_legend_handles_labels()
        axs[i].get_legend().remove()
    lgd = f.legend(handles = h, labels = l,
            bbox_to_anchor=(0.5, -0.01), loc='upper center', 
            ncol=2, prop={'size': 14})
    title = f.suptitle(ep, fontsize=14)
    plt.show()


In [ ]:
MI

In [14]:
MI = pd.DataFrame(single_val, index = ['real', 'shuffle']).T
MI.to_csv('data/output/mutual_info/OTS/estimated_mutual_info_all_runs.csv')